Event-based directional prediction on SPY using internal data + mlfinlab

This notebook:
- Fetches SPY OHLCV using findata
- Builds dollar bars (Lopez de Prado)
- Detects events via symmetric CUSUM
- Labels with the triple-barrier method
- Engineers compact features
- Trains a baseline classifier with time-based CV
- Produces asset_weights and asset_returns and runs backtest(asset_weights, asset_returns)

Notes: Dollar bars are constructed from daily OHLCV as a demonstration (using Close and Volume to proxy dollar value).

In [ ]:
# Imports and setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from datetime import datetime

# Data & labeling utilities
from mlfinlab.util.volatility import get_daily_vol
from mlfinlab.filters.filters import cusum_filter
from mlfinlab.labeling.labeling import add_vertical_barrier, get_events, get_bins
from mlfinlab.data_structures.standard_data_structures import get_dollar_bars

# Modeling
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score

# Plotting (optional)
import matplotlib.pyplot as plt

# Display options
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 50)

In [ ]:
# 1) Fetch SPY OHLCV via internal data wrapper
from findata.equity_prices import get_equity_prices

start_date = '2005-01-01'
end_date = None  # up to latest

ohlcv_df = get_equity_prices(
    tickers=['SPY'],
    start_date=start_date,
    end_date=end_date,
    interval='1d',
    auto_adjust=True,
    fields=['Open','High','Low','Close','Volume'],
    as_wide=True,
    debug=False,
)

# Collapse MultiIndex columns if present and keep SPY only
if isinstance(ohlcv_df.columns, pd.MultiIndex):
    ohlcv_df = ohlcv_df.xs('SPY', axis=1, level=0)

ohlcv_df = ohlcv_df.dropna().sort_index()
print(ohlcv_df.head())
print('\nShape:', ohlcv_df.shape)
ohlcv_df.tail()

In [ ]:
# 2) Build dollar bars from OHLCV (proxy using Close as price and Volume as size)
# Compute threshold as ~1/50 average daily dollar value, per LOP suggestion
adv = (ohlcv_df['Close'] * ohlcv_df['Volume']).mean()
threshold = adv / 50.0
print(f"Average daily $ volume: {adv:,.0f}; threshold (1/50): {threshold:,.0f}")

tick_like = pd.DataFrame({
    'date_time': ohlcv_df.index,
    'price': ohlcv_df['Close'].values,
    'volume': ohlcv_df['Volume'].values,
})

bars_df = get_dollar_bars(
    file_path_or_df=tick_like,
    threshold=threshold,
    batch_size=2_000_000,
    verbose=False,
    to_csv=False,
)

bars_df = bars_df.set_index('date_time').sort_index()
print(bars_df[['open','high','low','close','volume']].head())
print('\nBars shape:', bars_df.shape)
bars_df.tail()

In [ ]:
# 3) Volatility estimate on dollar-bar closes
close = bars_df['close'].astype(float)
sigma = get_daily_vol(close, lookback=100).dropna()
print('Volatility sample:')
print(sigma.dropna().iloc[:5])

# Align sigma to bars index
sigma = sigma.reindex(bars_df.index).fillna(method='ffill')

In [ ]:
# 4) CUSUM events on returns with threshold tied to volatility
rets = close.pct_change().fillna(0.0)
med_sig = float(np.nanmedian(sigma.values))
k_mult = 0.5
threshold = med_sig * k_mult
print(f"CUSUM threshold = median(sigma)*{k_mult} = {threshold:.6f}")

t_events = pd.DatetimeIndex(cusum_filter(rets, threshold=threshold, time_stamps=True))
print('Num events:', len(t_events))
print('First events:', t_events[:5])

In [ ]:
# 5) Triple barrier labeling
# Vertical barrier 5 bars ahead
vb_days = 5

t1 = add_vertical_barrier(
    t_events=pd.Series(index=t_events, dtype=float),
    close=close,
    num_days=vb_days,
)

pt_sl = [1.5, 1.5]
min_ret = float(sigma.quantile(0.25))
print('min_ret (25th pct of sigma):', min_ret)

events = get_events(
    close=close,
    t_events=t_events,
    pt_sl=pt_sl,
    target=sigma,
    min_ret=min_ret,
    num_threads=1,
    vertical_barrier_times=t1,
    side_prediction=None,
    verbose=False,
)

labels = get_bins(events, close=close)
print(labels.head())

# Keep binary label y
y = labels['bin'].dropna().astype(int)
# Align events to those with labels
evt_index = y.index
print('Labeled events:', len(evt_index))

In [ ]:
# 6) Compact feature set engineered on bars, aligned to event times
feat = pd.DataFrame(index=bars_df.index)
feat['ret_1'] = close.pct_change(1)
feat['ret_3'] = close.pct_change(3)
feat['ret_5'] = close.pct_change(5)
feat['vol_20'] = close.pct_change().rolling(20).std()
feat['mom_10'] = close.pct_change(10)

# RSI(14)
delta = close.pct_change()
up = delta.clip(lower=0).rolling(14).mean()
down = (-delta.clip(upper=0)).rolling(14).mean()
rs = up / (down.replace(0, np.nan))
feat['rsi_14'] = 100 - 100 / (1 + rs)

# Dollar volume z-score over a slow window
feat['dv'] = (close * bars_df['volume']).rolling(5).mean()
feat['dv'] = (feat['dv'] - feat['dv'].rolling(60).mean()) / feat['dv'].rolling(60).std()

# Align to labeled event times and clean
X = feat.reindex(evt_index).fillna(method='ffill').dropna()
y_aligned = y.reindex(X.index).dropna().astype(int)
X = X.reindex(y_aligned.index)

print(X.head())
print('\nFeature shape:', X.shape, 'Label shape:', y_aligned.shape)

In [ ]:
# 7) Baseline classifier with time-based CV and OOF predictions
pipe = Pipeline([
    ('scaler', StandardScaler(with_mean=True, with_std=True)),
    ('clf', LogisticRegression(C=1.0, class_weight='balanced', max_iter=200, solver='lbfgs')),
])

n_splits = 5
tss = TimeSeriesSplit(n_splits=n_splits)

proba_oof = pd.Series(index=X.index, dtype=float)
pred_oof = pd.Series(index=X.index, dtype=int)

fold_reports = []
for fold, (tr_idx, te_idx) in enumerate(tss.split(X), start=1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y_aligned.iloc[tr_idx], y_aligned.iloc[te_idx]

    pipe.fit(X_tr, y_tr)
    p_te = pipe.predict_proba(X_te)[:, 1]
    y_hat = (p_te >= 0.5).astype(int)

    proba_oof.iloc[te_idx] = p_te
    pred_oof.iloc[te_idx] = y_hat

    report = {
        'fold': fold,
        'roc_auc': roc_auc_score(y_te, p_te) if y_te.nunique() > 1 else np.nan,
        'accuracy': accuracy_score(y_te, y_hat),
        'precision': precision_score(y_te, y_hat, zero_division=0),
        'recall': recall_score(y_te, y_hat, zero_division=0),
        'n_train': len(tr_idx),
        'n_test': len(te_idx),
    }
    fold_reports.append(report)

cv_report = pd.DataFrame(fold_reports)
print(cv_report)
print('\nMean metrics:', cv_report[['roc_auc','accuracy','precision','recall']].mean())

# Fit final model on all data
final_model = pipe.fit(X, y_aligned)

# Keep for downstream
oof_proba = proba_oof.sort_index()
oof_pred = pred_oof.sort_index()

In [ ]:
# 8) Map event predictions to SPY weights and forward-fill between events
sig = (oof_proba >= 0.5).astype(float)
weights_events = pd.Series(sig.values, index=sig.index, name='SPY')

# Reindex to bar timeline and forward-fill; flat before first signal
asset_weights = weights_events.reindex(bars_df.index).ffill().fillna(0.0).to_frame()
asset_weights = asset_weights.rename_axis('date').rename(columns={'SPY':'SPY'})

print(asset_weights.head())
print('\nWeights tail:')
print(asset_weights.tail())

In [ ]:
# 9) Compute asset returns from prices and weights (execution lag = 1 bar)
bar_rets = close.pct_change().rename('SPY').to_frame()
execution_lag = 1
adj_weights = asset_weights.shift(execution_lag).fillna(0.0)

# Strategy returns by asset
asset_returns = adj_weights.values * bar_rets.values
asset_returns = pd.DataFrame(asset_returns, index=bar_rets.index, columns=['SPY'])

print(asset_returns.head())
print('\nStrategy return summary:')
print(asset_returns.describe())

In [ ]:
# 10) Final: call user-defined backtest once
# Expects a function backtest(asset_weights: pd.DataFrame, asset_returns: pd.DataFrame)
backtest(asset_weights, asset_returns)